# Laboratório — normal, z-score e normal multivariada

Este laboratório verifica probabilidades normais, padronização, diagnósticos gráficos e geometria multivariada. Também mostra dois limites importantes: padronização não produz normalidade e correlação zero não garante independência.

**Dependências:** Python ≥ 3.10, NumPy ≥ 1.26, SciPy ≥ 1.11 e Matplotlib ≥ 3.8.  
**Reprodutibilidade:** `numpy.random.default_rng(20260907)`.


## 1. Ambiente

Na SciPy, `norm` usa `loc` como média e `scale` como desvio-padrão. Na notação $\mathcal N(\mu,\sigma^2)$, o segundo termo é a variância.


In [1]:
import sys
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import norm, multivariate_normal, probplot, chi2, skew

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__} | SciPy: {scipy.__version__} | Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")

Python: 3.12.13
NumPy: 2.3.5 | SciPy: 1.17.0 | Matplotlib: 3.10.8
Seed: 20260907


## 2. Densidade, localização e escala

As curvas abaixo isolam os efeitos de mudar $\mu$ e $\sigma$. A integração numérica confirma que a área de cada densidade é aproximadamente 1.


In [2]:
x = np.linspace(-7, 9, 4000)
modelos = [(0, 1, "N(0, 1)"), (3, 1, "N(3, 1)"), (0, 2, "N(0, 4)")]

fig, ax = plt.subplots(figsize=(9, 4.5))
for mu_i, sigma_i, rotulo in modelos:
    pdf = norm.pdf(x, loc=mu_i, scale=sigma_i)
    area = np.trapezoid(pdf, x)
    print(f"{rotulo}: área na grade = {area:.8f}")
    assert area > 0.999
    ax.plot(x, pdf, linewidth=2, label=rotulo)
ax.set(xlabel="x", ylabel="Densidade", title="Média desloca; desvio-padrão altera a largura")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

N(0, 1): área na grade = 1.00000000
N(3, 1): área na grade = 1.00000000
N(0, 4): área na grade = 0.99976397


## 3. Exemplo da latência

Para $X\sim\mathcal N(72,8^2)$, calculamos o z-score de 84 ms e a cauda direita. `sf` calcula $P(X>x)$ diretamente.


In [3]:
mu, sigma, observado = 72.0, 8.0, 84.0
z = (observado - mu) / sigma
cauda = norm.sf(observado, loc=mu, scale=sigma)
cauda_padrao = norm.sf(z)

print(f"z-score: {z:.6f}")
print(f"P(X>84): {cauda:.9f}")

assert z == 1.5
assert np.isclose(cauda, 0.0668072013, atol=1e-10)
assert np.isclose(cauda, cauda_padrao, atol=1e-15)

z-score: 1.500000
P(X>84): 0.066807201


## 4. Regra 68–95–99,7 e quantis

Calculamos as áreas centrais exatas para 1, 2 e 3 desvios-padrão. Depois obtemos o quantil superior que delimita o intervalo central de 95%.


In [4]:
for k in [1, 2, 3]:
    central = norm.cdf(k) - norm.cdf(-k)
    print(f"P(|Z|<={k}) = {central:.9f}")

z_975 = norm.ppf(0.975)
print(f"Quantil 0,975: {z_975:.9f}")

assert np.isclose(norm.cdf(1) - norm.cdf(-1), 0.6826894921, atol=1e-10)
assert np.isclose(norm.cdf(2) - norm.cdf(-2), 0.9544997361, atol=1e-10)
assert np.isclose(z_975, 1.9599639845, atol=1e-10)

P(|Z|<=1) = 0.682689492
P(|Z|<=2) = 0.954499736
P(|Z|<=3) = 0.997300204
Quantil 0,975: 1.959963985


## 5. Simulação e padronização

Geramos 200 mil latências do modelo. A amostra padronizada deve ter média próxima de 0 e desvio-padrão próximo de 1; as proporções centrais devem acompanhar a teoria.


In [5]:
n_sim = 200_000
latencias = rng.normal(loc=mu, scale=sigma, size=n_sim)
zs = (latencias - mu) / sigma

print(f"Média de X: {latencias.mean():.5f} | DP de X: {latencias.std(ddof=0):.5f}")
print(f"Média de Z: {zs.mean():.6f} | DP de Z: {zs.std(ddof=0):.6f}")
for k in [1, 2, 3]:
    print(f"Cobertura empírica ±{k}σ: {np.mean(np.abs(zs) <= k):.6f}")

assert abs(latencias.mean() - mu) < 0.05
assert abs(latencias.std(ddof=0) - sigma) < 0.04
assert abs(zs.mean()) < 0.006
assert abs(zs.std(ddof=0) - 1) < 0.006
assert abs(np.mean(np.abs(zs) <= 2) - 0.9544997361) < 0.002

Média de X: 72.00108 | DP de X: 8.01183
Média de Z: 0.000135 | DP de Z: 1.001479
Cobertura empírica ±1σ: 0.681785
Cobertura empírica ±2σ: 0.954005
Cobertura empírica ±3σ: 0.997415


In [6]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(latencias, bins=70, density=True, alpha=0.55, color="#2563eb", label="Simulação")
grade = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
ax.plot(grade, norm.pdf(grade, loc=mu, scale=sigma), color="#dc2626", linewidth=2.5, label="PDF teórica")
ax.axvline(observado, color="#111827", linestyle="--", label="84 ms")
ax.set(xlabel="Latência (ms)", ylabel="Densidade", title="Normal(72, 8²): simulação × teoria")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 6. Padronizar não cria normalidade

Geramos dados lognormais, que são assimétricos, e os padronizamos com sua média e seu desvio-padrão amostrais. Média e escala mudam; a assimetria permanece.


In [7]:
assimetrica = rng.lognormal(mean=0.0, sigma=0.8, size=50_000)
assimetrica_z = (assimetrica - assimetrica.mean()) / assimetrica.std(ddof=0)

skew_original = skew(assimetrica, bias=False)
skew_z = skew(assimetrica_z, bias=False)
print(f"Antes: média={assimetrica.mean():.6f}, DP={assimetrica.std(ddof=0):.6f}, assimetria={skew_original:.6f}")
print(f"Depois: média={assimetrica_z.mean():.6f}, DP={assimetrica_z.std(ddof=0):.6f}, assimetria={skew_z:.6f}")

assert abs(assimetrica_z.mean()) < 1e-12
assert abs(assimetrica_z.std(ddof=0) - 1) < 1e-12
assert np.isclose(skew_original, skew_z, atol=1e-12)
assert skew_z > 2

Antes: média=1.377693, DP=1.309855, assimetria=4.387252
Depois: média=0.000000, DP=1.000000, assimetria=4.387252


## 7. Gráfico de probabilidade normal

`probplot` compara observações ordenadas com quantis normais teóricos. Linearidade é compatível com normalidade; desvios sistemáticos nas pontas revelam diferenças de cauda ou assimetria.


In [8]:
rng_diag = np.random.default_rng(SEED + 1)
amostra_normal = rng_diag.normal(size=3_000)
amostra_lognormal = rng_diag.lognormal(mean=0, sigma=0.8, size=3_000)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
res_normal = probplot(amostra_normal, dist="norm", plot=axes[0])
res_log = probplot(amostra_lognormal, dist="norm", plot=axes[1])
axes[0].set_title("Dados normais")
axes[1].set_title("Dados lognormais")
plt.tight_layout()
plt.show()

r_normal = res_normal[1][2]
r_log = res_log[1][2]
print(f"Correlação do ajuste — normal: {r_normal:.6f} | lognormal: {r_log:.6f}")
assert r_normal > 0.999
assert r_log < 0.95

Correlação do ajuste — normal: 0.999351 | lognormal: 0.874710


## 8. Parâmetros do treino e mudança no teste

O teste foi gerado com média deslocada. Aplicar os parâmetros do treino mantém o deslocamento visível; recalcular no teste o apagaria e ainda usaria informação externa ao ajuste.


In [9]:
rng_split = np.random.default_rng(SEED + 2)
treino = rng_split.normal(10, 2, size=20_000)
teste = rng_split.normal(13, 2, size=5_000)

media_treino = treino.mean()
dp_treino = treino.std(ddof=1)
teste_z_correto = (teste - media_treino) / dp_treino
teste_z_vazado = (teste - teste.mean()) / teste.std(ddof=1)

print(f"Teste com parâmetros do treino — média z: {teste_z_correto.mean():.4f}")
print(f"Teste recentralizado em si mesmo — média z: {teste_z_vazado.mean():.4f}")

assert 1.43 < teste_z_correto.mean() < 1.57
assert abs(teste_z_vazado.mean()) < 1e-12

Teste com parâmetros do treino — média z: 1.4962
Teste recentralizado em si mesmo — média z: -0.0000


## 9. Normal multivariada: parâmetros válidos

Usaremos média $[2,-1]^T$ e covariância $[[4,3],[3,9]]$. Autovalores positivos confirmam que a matriz é positiva definida; a correlação teórica é 0,5.


In [10]:
mu_vec = np.array([2.0, -1.0])
Sigma = np.array([[4.0, 3.0], [3.0, 9.0]])

assert np.allclose(Sigma, Sigma.T)
autovalores = np.linalg.eigvalsh(Sigma)
rho = Sigma[0, 1] / np.sqrt(Sigma[0, 0] * Sigma[1, 1])

print(f"Autovalores: {autovalores}")
print(f"Correlação teórica: {rho:.6f}")
assert np.all(autovalores > 0)
assert rho == 0.5

Autovalores: [ 2.59487516 10.40512484]
Correlação teórica: 0.500000


## 10. Simulação bivariada e recuperação dos parâmetros

Com 200 mil vetores, média e covariância amostrais devem se aproximar dos parâmetros geradores.


In [11]:
rng_multi = np.random.default_rng(SEED + 3)
amostra_multi = rng_multi.multivariate_normal(mu_vec, Sigma, size=200_000)
media_emp = amostra_multi.mean(axis=0)
cov_emp = np.cov(amostra_multi, rowvar=False, ddof=1)

print("Média empírica:", np.round(media_emp, 5))
print("Covariância empírica:\n", np.round(cov_emp, 5))

assert np.allclose(media_emp, mu_vec, atol=0.02)
assert np.allclose(cov_emp, Sigma, atol=0.06)

Média empírica: [ 1.98564 -1.01534]
Covariância empírica:
 [[4.00924 3.02447]
 [3.02447 9.0383 ]]


## 11. Contornos e elipse de 95%

Em duas dimensões, a fronteira de 95% usa $D_M^2=\chi^2_{2;0,95}$. Os eixos da elipse vêm dos autovalores; sua orientação vem dos autovetores.


In [12]:
limiar_95 = chi2.ppf(0.95, df=2)
valores, vetores = np.linalg.eigh(Sigma)
ordem = valores.argsort()[::-1]
valores, vetores = valores[ordem], vetores[:, ordem]
angulo = np.degrees(np.arctan2(vetores[1, 0], vetores[0, 0]))
largura, altura = 2 * np.sqrt(valores * limiar_95)

fig, ax = plt.subplots(figsize=(7, 6))
sub = amostra_multi[:4_000]
ax.scatter(sub[:, 0], sub[:, 1], s=7, alpha=0.18, color="#2563eb", label="Amostra")
elipse = Ellipse(mu_vec, width=largura, height=altura, angle=angulo,
                  fill=False, linewidth=2.5, color="#dc2626", label="Região teórica de 95%")
ax.add_patch(elipse)
ax.scatter(*mu_vec, marker="x", s=100, linewidth=3, color="#111827", label="Média")
ax.set(xlabel="X₁", ylabel="X₂", title="Normal bivariada e elipse de Mahalanobis")
ax.set_aspect("equal", adjustable="box")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print(f"Limiar qui-quadrado de 95%: {limiar_95:.9f}")
assert np.isclose(limiar_95, 5.9914645471, atol=1e-10)

Limiar qui-quadrado de 95%: 5.991464547


## 12. Distância de Mahalanobis e cobertura

Calculamos a distância do ponto $[4,2]^T$ e, depois, a cobertura da região em toda a simulação. Usamos `solve` em vez de formar a inversa explicitamente.


In [13]:
ponto = np.array([4.0, 2.0])
delta_ponto = ponto - mu_vec
d2_ponto = delta_ponto @ np.linalg.solve(Sigma, delta_ponto)

deltas = amostra_multi - mu_vec
d2_amostra = np.einsum("ni,ni->n", deltas, np.linalg.solve(Sigma, deltas.T).T)
cobertura = np.mean(d2_amostra <= limiar_95)

print(f"D² do ponto [4, 2]: {d2_ponto:.9f} | D: {np.sqrt(d2_ponto):.9f}")
print(f"Cobertura empírica da elipse de 95%: {cobertura:.6f}")

assert np.isclose(d2_ponto, 4/3, atol=1e-12)
assert abs(cobertura - 0.95) < 0.002

D² do ponto [4, 2]: 1.333333333 | D: 1.154700538
Cobertura empírica da elipse de 95%: 0.949380


## 13. Cholesky: de normais independentes para correlacionadas

Se $\Sigma=LL^T$ e $Z\sim\mathcal N(0,I)$, então $\mu+LZ$ possui a covariância desejada.


In [14]:
L = np.linalg.cholesky(Sigma)
assert np.allclose(L @ L.T, Sigma)

rng_chol = np.random.default_rng(SEED + 4)
Z = rng_chol.normal(size=(150_000, 2))
X_chol = mu_vec + Z @ L.T
cov_chol = np.cov(X_chol, rowvar=False, ddof=1)

print("L =\n", np.round(L, 6))
print("Covariância via Cholesky =\n", np.round(cov_chol, 5))
assert np.allclose(X_chol.mean(axis=0), mu_vec, atol=0.02)
assert np.allclose(cov_chol, Sigma, atol=0.07)

L =
 [[2.       0.      ]
 [1.5      2.598076]]
Covariância via Cholesky =
 [[3.99851 3.00807]
 [3.00807 9.03771]]


## 14. Covariância zero com dependência

Tomamos $U$ uniforme e $Y=U^2-1/3$. A correlação é aproximadamente zero por simetria, mas $Y$ depende deterministicamente de $U$. Médias condicionais em faixas de $|U|$ tornam isso evidente.


In [15]:
rng_dep = np.random.default_rng(SEED + 5)
U = rng_dep.uniform(-1, 1, size=300_000)
Y = U**2 - 1/3
corr = np.corrcoef(U, Y)[0, 1]
media_centro = Y[np.abs(U) < 0.2].mean()
media_extremos = Y[np.abs(U) > 0.8].mean()

print(f"Correlação(U, Y): {corr:.6f}")
print(f"E[Y | |U|<0,2]: {media_centro:.6f}")
print(f"E[Y | |U|>0,8]: {media_extremos:.6f}")

assert abs(corr) < 0.005
assert media_centro < -0.30
assert media_extremos > 0.30

Correlação(U, Y): -0.001166
E[Y | |U|<0,2]: -0.320007
E[Y | |U|>0,8]: 0.479799


## 15. Caudas e estabilidade numérica

Para $z=10$, `1-cdf` arredonda para zero em ponto flutuante comum, enquanto `sf` ainda representa a cauda.


In [16]:
z_extremo = 10.0
cauda_subtracao = 1 - norm.cdf(z_extremo)
cauda_sf = norm.sf(z_extremo)

print(f"1 - CDF(10): {cauda_subtracao:.3e}")
print(f"SF(10): {cauda_sf:.12e}")

assert cauda_subtracao == 0.0
assert np.isclose(cauda_sf, 7.6198530242e-24, rtol=1e-10)

1 - CDF(10): 0.000e+00
SF(10): 7.619853024160e-24


## 16. Checagens finais

As asserções condensam os invariantes: padronização exata sob parâmetros conhecidos, covariância válida, identidade de Cholesky e correspondência entre CDF e quantil.


In [17]:
assert np.isclose(norm.cdf(norm.ppf(0.12345)), 0.12345)
assert np.all(np.linalg.eigvalsh(Sigma) > 0)
assert np.allclose(L @ L.T, Sigma)
assert np.isclose(norm.sf(1.5), cauda)
assert abs(cobertura - 0.95) < 0.002

print("Todas as verificações numéricas foram concluídas com sucesso.")

Todas as verificações numéricas foram concluídas com sucesso.


## Conclusões

- `loc` é a média e `scale` é o desvio-padrão na SciPy.
- O z-score mede distância em unidades de desvio-padrão; não corrige a forma da distribuição.
- Dados simulados confirmaram probabilidades, quantis e coberturas normais.
- Ajustar o pré-processamento apenas no treino preservou um deslocamento real no teste.
- A covariância controlou a escala e a orientação da nuvem bivariada.
- A região multivariada de 95% usou o quantil $\chi^2_2$, não 1,96.
- Cholesky gerou amostras com a covariância planejada.
- O contraexemplo mostrou dependência mesmo com correlação próxima de zero.

Próximo passo: Lei dos Grandes Números, Teorema Central do Limite e Monte Carlo na Aula 10.
